# Pronunciation Assessment — real-time endpoint

This product assesses, in real time, how closely a recording matches the text a speaker was supposed to say,
in French and Arabic.

---

## About this notebook

This notebook runs the whole lifecycle:
* deploy a real-time endpoint from the model package,
* send a recording
* read the assessment
* delete the endpoint again.

But you need to
* manage your notebook (we recommend Sagemaker Studio's notebook)
* subscribe on your own (see below)

## Before you start

1. **Subscribe** to Pronunciation Assessment on AWS Marketplace and accept the
   EULA.
   * https://aws.amazon.com/marketplace/pp/prodview-uzfcfzqcmds7i
2. **Check the Model Package ARN.** The notebook already carries the ARNs of
   version 1.0.2 for every supported region, and section 2 picks the one
   matching your session's region — so there is usually nothing to edit.

   Paste into `MODEL_PACKAGE_ARN_BY_REGION` only if your subscription shows a
   different ARN. To find it on the listing:
   * On the subscription page, click **Configure**
   * Select one of the 3 options — nothing is launched at that point
   * Read the Model Package ARN at the bottom right of the page
3. Run this notebook with an IAM role that can create SageMaker models and
   endpoints (`AmazonSageMakerFullAccess` is enough).

## Cost
This notebook creates a real endpoint. Real-time software charges are per
request, so an idle endpoint costs you only its instance — but that instance
bills by the hour for as long as the endpoint exists. The last section deletes
it. Run that section even if something fails partway through.


---
## 1. Install dependencies

`requests-toolbelt` builds the multipart body and, crucially, reports the exact
`Content-Type` that matches it.

In [1]:
%pip install -q "sagemaker>=2.200,<3" "boto3>=1.34" "requests-toolbelt>=1.0"

Note: you may need to restart the kernel to use updated packages.


---
## 2. Configure

The Model Package ARNs below ship with the notebook — one per supported region,
for version 1.0.2 — and the next section picks the one matching your session's
region. Replace an entry only if your own subscription shows a different ARN for
your region.

In [2]:
# The ARNs of the latest version of the product (1.0.2), one per region.
# Replace an entry only if your own subscription shows a different ARN.
MODEL_PACKAGE_ARN_BY_REGION = {
  "ap-south-1": "arn:aws:sagemaker:ap-south-1:077584701553:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "eu-west-3": "arn:aws:sagemaker:eu-west-3:843114510376:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "eu-north-1": "arn:aws:sagemaker:eu-north-1:136758871317:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "eu-west-2": "arn:aws:sagemaker:eu-west-2:856760150666:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "eu-west-1": "arn:aws:sagemaker:eu-west-1:985815980388:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "ap-northeast-2": "arn:aws:sagemaker:ap-northeast-2:745090734665:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "ap-northeast-1": "arn:aws:sagemaker:ap-northeast-1:977537786026:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "ca-central-1": "arn:aws:sagemaker:ca-central-1:470592106596:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "sa-east-1": "arn:aws:sagemaker:sa-east-1:270155090741:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "ap-southeast-1": "arn:aws:sagemaker:ap-southeast-1:192199979996:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "ap-southeast-2": "arn:aws:sagemaker:ap-southeast-2:666831318237:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "eu-central-1": "arn:aws:sagemaker:eu-central-1:446921602837:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "us-east-1": "arn:aws:sagemaker:us-east-1:865070037744:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "us-east-2": "arn:aws:sagemaker:us-east-2:057799348421:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "us-west-1": "arn:aws:sagemaker:us-west-1:382657785993:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7",
  "us-west-2": "arn:aws:sagemaker:us-west-2:594846645681:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7"
}
INSTANCE_TYPE = "ml.c5.large"  # ml.m5.large is also supported
ENDPOINT_NAME = "ekinox-pronunciation-assessment-demo"

In [3]:
import os

# The v2 SDK prints a v2 -> v3 deprecation banner on import. Section 1 pins v2
# on purpose: v3 removed both of the APIs used here, `sagemaker.Session` and
# `get_execution_role()`. So the banner is silenced rather than acted on. This
# must be set before `import sagemaker`, and only takes effect on a kernel that
# has not imported it yet.
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

import sagemaker

session = sagemaker.Session()
region = session.boto_region_name

if region not in MODEL_PACKAGE_ARN_BY_REGION:
    raise RuntimeError(
        f"This product is not available in {region}. "
        f"Available: {', '.join(MODEL_PACKAGE_ARN_BY_REGION)}"
    )

MODEL_PACKAGE_ARN = MODEL_PACKAGE_ARN_BY_REGION[region]

try:
    # Works inside SageMaker Studio and notebook instances.
    role = sagemaker.get_execution_role()
except ValueError:
    # Running locally: set this to a role SageMaker can assume.
    role = "arn:aws:iam::<your-account-id>:role/<your-sagemaker-execution-role>"

print(f"Region:        {region}")
print(f"Role:          {role}")
print(f"Model package: {MODEL_PACKAGE_ARN}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region:        eu-north-1
Role:          arn:aws:iam::<your-account-id>:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole
Model package: arn:aws:sagemaker:eu-north-1:136758871317:model-package/ekinox-pronunciation-1-0-2-b8f87900ed6133ef869eb6f6925cf2b7


---
## 3. Deploy the endpoint

This takes a few minutes. The container loads its model on start-up, so the
endpoint reports `InService` shortly before it is fully warm — the first request
may briefly return `503`. Section 5 retries for exactly that reason.

In [4]:
from botocore.exceptions import ClientError

sagemaker_client = session.boto_session.client("sagemaker")

def remove_endpoint_resources():
    """Delete the endpoint, endpoint config and model named ENDPOINT_NAME.

    The three share a name but are independent resources: deleting the endpoint
    leaves the config and the model behind, and CreateModel /
    CreateEndpointConfig refuse to overwrite an existing name. Nothing is
    reserved after a delete, so clearing them makes a redeploy possible
    straight away.

    Returns the labels of whatever was actually deleted, so this is safe to run
    before a deploy and again during clean-up.
    """
    deleted = []
    for label, delete in (
        ("endpoint", lambda: sagemaker_client.delete_endpoint(
            EndpointName=ENDPOINT_NAME)),
        ("endpoint config", lambda: sagemaker_client.delete_endpoint_config(
            EndpointConfigName=ENDPOINT_NAME)),
        ("model", lambda: sagemaker_client.delete_model(
            ModelName=ENDPOINT_NAME)),
    ):
        try:
            delete()
        except ClientError as error:
            # "Could not find ..." is the expected case when there is nothing
            # to remove. Anything else — a missing permission, most likely —
            # should not be swallowed.
            if error.response["Error"]["Code"] != "ValidationException":
                raise
            continue

        print(f"Deleted {label}: {ENDPOINT_NAME}")
        deleted.append(label)

        if label == "endpoint":
            # Endpoints delete asynchronously and hold their name until the
            # deletion completes. Configs and models delete immediately.
            sagemaker_client.get_waiter("endpoint_deleted").wait(
                EndpointName=ENDPOINT_NAME)

    return deleted


# Clear anything left by an earlier run, so this cell can be re-run after a
# deploy that failed partway.
remove_endpoint_resources()

created_model = sagemaker_client.create_model(
    ModelName=ENDPOINT_NAME,
    ExecutionRoleArn=role,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    # Marketplace model packages are required to run without network access.
    # ModelPackage.deploy() set this for you; the API does not.
    EnableNetworkIsolation=True,
)
print(f"Model:           {created_model['ModelArn']}")

created_config = sagemaker_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": ENDPOINT_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
        }
    ],
)
print(f"Endpoint config: {created_config['EndpointConfigArn']}")

created_endpoint = sagemaker_client.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_NAME,
)
print(f"Endpoint:        {created_endpoint['EndpointArn']}")
print(f"Instance type:   {INSTANCE_TYPE}")

Model:           arn:aws:sagemaker:eu-north-1:<your-account-id>:model/ekinox-pronunciation-assessment-demo
Endpoint config: arn:aws:sagemaker:eu-north-1:<your-account-id>:endpoint-config/ekinox-pronunciation-assessment-demo
Endpoint:        arn:aws:sagemaker:eu-north-1:<your-account-id>:endpoint/ekinox-pronunciation-assessment-demo
Instance type:   ml.c5.large


In [5]:
import time

print("Waiting for the endpoint to come up — this takes a few minutes...")

last_status = None
deadline = time.monotonic() + 30 * 60

while time.monotonic() < deadline:
    description = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = description["EndpointStatus"]

    if status != last_status:
        print(f"  {status}")
        last_status = status

    if status == "InService":
        break
    if status in ("Failed", "OutOfService"):
        raise RuntimeError(
            f"Endpoint {ENDPOINT_NAME} did not come up (status {status}): "
            f"{description.get('FailureReason', 'no reason reported')}. "
            f"Run section 9 to clean up before retrying."
        )

    time.sleep(15)
else:
    raise RuntimeError(f"Endpoint {ENDPOINT_NAME} still not ready after 30 minutes.")

print(f"Endpoint ready: {ENDPOINT_NAME}")

Waiting for the endpoint to come up — this takes a few minutes...
  Creating
  InService
Endpoint ready: ekinox-pronunciation-assessment-demo


---
## 4. Build the request

Three parts, all required:

| Part | Type | Description |
|---|---|---|
| `language` | text | `fr`, `fr-fr`, `fr_fr` for French; `ar`, `ar-sa`, `ar_sa` for Arabic |
| `audio` | file | The recording, as a WAV file |
| `expectedText` | text | What the speaker was supposed to say |

`MultipartEncoder.content_type` is the piece that matters: it is the
`multipart/form-data; boundary=...` string describing the body the encoder just
built. Pass that same string as `ContentType` and the two always agree.

In [6]:
import os

from requests_toolbelt.multipart.encoder import MultipartEncoder


def build_request(audio_path, expected_text, language="fr-fr"):
    """Return (body_bytes, content_type) for one assessment request."""
    with open(audio_path, "rb") as audio_file:
        encoder = MultipartEncoder(
            fields={
                "language": language,
                "expectedText": expected_text,
                # The filename and content type are ignored by the model, but
                # most HTTP libraries will not emit a file part without them.
                "audio": (os.path.basename(audio_path), audio_file, "audio/wav"),
            }
        )
        # Read the body before the file closes.
        return encoder.to_string(), encoder.content_type

---
## 5. Invoke

The sample recording is a French speaker attempting *"Les enfants ont mangé une petite tarte aux pommes."*
It ships with this repository in [`../data/sample_audio.wav`](../data/sample_audio.wav).

It is not a clean read — expect `NOT ACCEPTED`, with `ont` omitted; `petite`,
`tarte` and `aux` mispronounced; and an extra word at the end reported as an
`insertion`. That is the sample doing its job: it exercises the fault paths
rather than returning a perfect score.

In [7]:
import json
import time

from botocore.exceptions import ClientError

runtime = session.sagemaker_runtime_client


def assess(audio_path, expected_text, language="fr-fr", attempts=12):
    """Invoke the endpoint, retrying while the model is still warming up."""
    body, content_type = build_request(audio_path, expected_text, language)

    for attempt in range(1, attempts + 1):
        try:
            response = runtime.invoke_endpoint(
                EndpointName=ENDPOINT_NAME,
                ContentType=content_type,
                Accept="application/json",
                Body=body,
            )
            return json.loads(response["Body"].read())
        except ClientError as error:
            # 503 means the model has not finished loading.
            status = error.response.get("OriginalStatusCode")
            if status != 503 or attempt == attempts:
                raise
            # Cap the backoff: doubling indefinitely would spend the
            # whole budget asleep. 12 attempts caps out near two minutes.
            wait = min(2 ** attempt, 15)
            print(f"Model still warming up, retrying in {wait}s...")
            time.sleep(wait)


result = assess("../data/sample_audio.wav", "Les enfants ont mangé une petite tarte aux pommes.", language="fr-fr")
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "language": "fr",
  "accepted": false,
  "scorePercent": 65.32093887259363,
  "faults": [
    "omission",
    "substitution"
  ],
  "assessmentVersion": "french-core-v4",
  "words": [
    {
      "id": "w0",
      "referenceWord": "Les",
      "producedText": "lez",
      "status": "correct",
      "faults": [],
      "scorePercent": 99.9559045038363,
      "phonemes": [
        {
          "referencePhoneme": "l",
          "producedPhoneme": "l",
          "scorePercent": 99.99752250524446,
          "status": "correct",
          "tolerated": false,
          "graphemes": {
            "start": 0,
            "end": 1
          }
        },
        {
          "referencePhoneme": "e",
          "producedPhoneme": "e",
          "scorePercent": 99.95332829575254,
          "status": "correct",
          "tolerated": false,
          "graphemes": {
            "start": 1,
            "end": 3
          }
        },
        {
          "referencePhoneme": "z",
          "producedPh

---
## 6. Read the result

The top level tells you whether the utterance passed and how it scored. Each
entry in `words` carries its own status, score, and phoneme detail, plus
`mispronouncedGraphemes` — character ranges into that word's `referenceWord`
that you can use to highlight the problem directly in your UI.

`status` is one of `correct`, `mispronounced`, `omitted`, or `insertion`.
Phonemes are IPA.

**`scorePercent` is a real 0–100 percentage**, at every level, higher is better.
But do not threshold on it to decide whether an attempt passed: it is a
confidence readout, best used to show a learner how close they were. Whether the
attempt is acceptable is already decided for you — `accepted` for the utterance,
`status` per word. See [`../docs/api.md`](../docs/api.md#scoring).

In [8]:
import unicodedata


def pad(value, width):
    """Pad `value` on the right, to `width` display columns.

    IPA uses combining marks — the nasal vowel `ɔ̃` is two code points but one
    glyph — so plain str.ljust over-pads anything carrying a diacritic. Count
    only the characters that actually occupy a column.
    """
    text = str(value)
    visible = sum(1 for char in text if not unicodedata.combining(char))
    return text + " " * max(0, width - visible)


def summarize(result):
    verdict = "ACCEPTED" if result["accepted"] else "NOT ACCEPTED"
    print(f"{verdict}  —  score {result['scorePercent']:.1f}%")
    print(f"language: {result['language']}    model: {result['assessmentVersion']}")
    if result["faults"]:
        print(f"faults:   {', '.join(result['faults'])}")

    print()
    print(pad("word", 16) + pad("status", 16) + pad("score", 12) + "produced")
    print("-" * 64)
    for word in result["words"]:
        score = word["scorePercent"]
        print(
            pad(word["referenceWord"] or "—", 16)
            + pad(word["status"], 16)
            + pad("—" if score is None else f"{score:.1f}%", 12)
            + (word["producedText"] or "—")
        )


summarize(result)

NOT ACCEPTED  —  score 65.3%
language: fr    model: french-core-v4
faults:   omission, substitution

word            status          score       produced
----------------------------------------------------------------
Les             correct         100.0%      lez
enfants         correct         80.5%       ɑ̃fɑ̃
ont             omitted         0.0%        —
mangé           correct         76.7%       mɑ̃ʒɛ
une             correct         100.0%      yn
petite          mispronounced   60.1%       pati
tarte           mispronounced   75.0%       tat
aux             mispronounced   2.6%        u
pommes.         correct         93.1%       pɔm
—               insertion       —           ʁuʒ


In [9]:
def show_phonemes(word):
    """Per-phoneme detail for one word, as returned in `words[].phonemes`."""
    print(f"{word['referenceWord'] or '—'}  ({word['status']})")
    print("  " + pad("expected", 12) + pad("produced", 12)
          + pad("score", 12) + pad("status", 16) + "tolerated")
    for phoneme in word["phonemes"]:
        print(
            "  "
            + pad(phoneme["referencePhoneme"] or "—", 12)
            + pad(phoneme["producedPhoneme"] or "—", 12)
            + pad(f"{phoneme['scorePercent']:.1f}%", 12)
            + pad(phoneme["status"], 16)
            + ("yes" if phoneme["tolerated"] else "no")
        )


# Show detail for the first mispronounced word. An `omitted` word has a single
# empty row to show, and an `insertion` has no reference word at all — its
# `phonemes` is empty by contract — so neither makes a useful example.
mispronounced = [w for w in result["words"] if w["status"] == "mispronounced"]
if mispronounced:
    show_phonemes(mispronounced[0])
else:
    print("No word was mispronounced.")

petite  (mispronounced)
  expected    produced    score       status          tolerated
  p           p           100.0%      correct         no
  ə           a           5.5%        correct         no
  t           t           95.1%       correct         no
  i           i           100.0%      correct         no
  t           —           0.0%        omitted         no


---
## 7. Try your own audio

Point `assess()` at any WAV file. `expectedText` is what the speaker was
supposed to say — the model scores the recording against it, so the two must
correspond.

**Audio must be 16-bit PCM in a RIFF/WAVE container.** Sample rate and channel
count are free — audio is resampled to 16 kHz mono internally — but anything
that is not PCM16 WAV (MP3, AAC, Opus, FLAC, 8/24-bit, 32-bit float) comes back
as a `400`. Convert first:

```
ffmpeg -i input.m4a -c:a pcm_s16le output.wav
```

SageMaker caps an `InvokeEndpoint` request at 6 MB — roughly three minutes at
16 kHz mono, about 35 seconds at 44.1 kHz stereo — and separately gives up on a
response after 60 seconds, which can bind first on a long recording. This model
is built for single utterances — a word, a phrase, a sentence — not long
recordings.

In [10]:
# result = assess("my-recording.wav", "Bonjour tout le monde", language="fr")
# summarize(result)

---
## 8. Errors

Failures come back as `ModelError`, with the model's own status and JSON body
attached:

| `OriginalStatusCode` | Meaning |
|---|---|
| `400` | The body is multipart but unusable: the `ContentType` carries no `boundary`, the body does not parse against the boundary it does carry, a part is missing, `expectedText` is blank, the language is unsupported, or the audio is empty or not decodable PCM16 WAV |
| `415` | The `ContentType` is not `multipart/form-data` at all. Note the difference from a `400`: a multipart request missing its `boundary` is a `400` — the media type is right, the request is malformed |
| `503` | The model is not ready yet — retry with backoff, as `assess()` does above |
| `500` | An unexpected internal failure |

The cell below sends an unsupported language to show the shape of a `400`.

In [11]:
try:
    assess("../data/sample_audio.wav", "Guten Tag", language="de")
except ClientError as error:
    print(f"status:  {error.response.get('OriginalStatusCode')}")
    print(f"message: {error.response.get('OriginalMessage')}")

status:  400
message: {"error":"Unsupported language 'de'."}


---
## 9. Clean up

**Run this.** The endpoint's instance bills by the hour until the endpoint is
deleted, whether or not you are sending it requests. It deletes by name, so it
works even if an earlier cell failed, and it is safe to run twice.

In [12]:
# Deletes by name, so clean-up works even when an earlier cell failed before
# the endpoint finished coming up. Safe to run twice.
if not remove_endpoint_resources():
    print(f"Nothing to delete — {ENDPOINT_NAME} is already cleaned up.")

Deleted endpoint: ekinox-pronunciation-assessment-demo
Deleted endpoint config: ekinox-pronunciation-assessment-demo
Deleted model: ekinox-pronunciation-assessment-demo


---
## Where to go next

- [`../docs/api.md`](../docs/api.md) — every request and response field
- [`../docs/openapi.yaml`](../docs/openapi.yaml) — the same contract as OpenAPI 3.0
- [`../data/sample_request.md`](../data/sample_request.md) — the request above, byte for byte
- [`../README.md`](../README.md) — supported languages, instance types, versioning

**Batch Transform is supported too**, for reprocessing a corpus you already
hold — one utterance per S3 object, `SplitType: None`. The catch is that
SageMaker sets `ContentType` once for a whole job and the multipart boundary
lives inside it, so every input object of a job has to be built with the *same*
boundary. [`../docs/api.md`](../docs/api.md#batch-transform) has the mechanics.

When latency matters — a speaker waiting on the result — keep calling a
real-time endpoint, concurrently if you need throughput.

Questions about the product go through the support channel on the AWS
Marketplace listing. Problems with this notebook:
[open an issue](https://github.com/EkinoxIO/ekinox-marketplace/issues).